# Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import confusion_matrix, classification_report, adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import LeaveOneOut
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors

from scipy.signal import find_peaks
from scipy.integrate import trapezoid
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
from scipy.stats import skew, kurtosis

warnings.filterwarnings('ignore')

DIR_DATA = os.getcwd()+"/data/"
DIR_OUTPUT = os.getcwd()+"/output/"

# Carregando dados

In [ ]:
base_name = "Crystallizer #1.csv"

df_dataset = pd.read_csv(DIR_DATA + base_name, sep=";", decimal=".")
df_dataset["TIMESTAMP"] = pd.to_datetime(df_dataset["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_dataset["Resultado de Ferro (ppm)"] = pd.to_numeric(df_dataset["Resultado de Ferro (ppm)"], errors="coerce")
df_dataset.sort_values(by="TIMESTAMP", inplace=True)
df_dataset

In [ ]:
df_duplicados = df_dataset[df_dataset.duplicated(subset=['Labref'], keep=False)]
df_duplicados

In [ ]:
# Retirando linhas 23 e 2141 que estão duplicadas mas não possuem amostras significativas
df_dataset.drop([23,2141], inplace=True) 
# Aplicando média para medidas com Labref iguais
agg_logic = {col: 'mean' if df_dataset[col].dtype.kind in 'biufc' else 'first' 
             for col in df_dataset.columns if col != 'Labref'}
df_dataset = df_dataset.groupby('Labref', as_index=False).agg(agg_logic)
df_dataset

# Carregando eventos identificados

In [ ]:
base_name_eventos = "Eventos-Reator1.csv"

df_eventos = pd.read_csv(DIR_DATA + base_name_eventos, sep=";", decimal=".")
df_eventos["TIMESTAMP"] = pd.to_datetime(df_eventos["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_eventos["Real"] = 1
df_eventos

# Definindo janela anterior aos eventos

In [ ]:
DIAS_JANELA   = 7  # dias da janela anterior ao evento
DIAS_BASELINE = 30   # dias de histórico antes da janela para contexto

# Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema

In [ ]:
# Filtra ultrapassagens
threshold = 5

df_over = df_dataset[df_dataset["Resultado de Ferro (ppm)"] > threshold].copy()
df_over = df_over.sort_values("TIMESTAMP")

# Mantém apenas a primeira ultrapassagem de cada grupo consecutivo
eventos_detectados = []
last_date = None

for _, row in df_over.iterrows():
    if last_date is None or (row["TIMESTAMP"] - last_date).days >= DIAS_JANELA:
        eventos_detectados.append(row["TIMESTAMP"])
        last_date = row["TIMESTAMP"]

# Cria dataframe com os eventos detectados
df_novos_eventos = pd.DataFrame({
    "TIMESTAMP": eventos_detectados,
    "Evento": "Ultrapassagem Fe > 5ppm mas sem problema relatado",
    "Real": 0
})

# Adiciona ao df_eventos existente
df_eventos = pd.concat([df_eventos, df_novos_eventos], ignore_index=True)
df_eventos = df_eventos.sort_values("TIMESTAMP").reset_index(drop=True)

df_eventos.drop_duplicates(subset=["TIMESTAMP"], keep='first', inplace=True)
df_eventos

# Plotando Gráfico das medições
Linhas vermelhas = Eventos relatados  
Linhas azuis = Eventos de ultapassagem sem relatos

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_dataset['TIMESTAMP'],
    y=df_dataset["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig.update_layout(
    template='plotly_white',
    hovermode='x unified'
)
fig.show()

In [ ]:
# fig.write_html("Grafico1.html")

## Gráfico com Média Móvel
Verificando se há tendência clara nos dados

In [ ]:
df_mm = df_dataset.copy()
num_amostras = 7
df_mm['Resultado_MM'] = df_mm["Resultado de Ferro (ppm)"].rolling(window=num_amostras).mean()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_mm['TIMESTAMP'],
    y=df_mm['Resultado_MM'],
    mode='lines',
    name="Média Móvel",
    line=dict(color='black')
))
fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    # Anotação separada (opcional)
    fig.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig.update_layout(
    template='plotly_white',
    hovermode='x unified'
)
fig.show()

# Clusterização (Não supervisionada)

### Taxas de Variação (ppm/hora)

**`taxa_max`**
A maior velocidade de subida (ou descida) que o ferro atingiu na janela. Captura se houve algum momento de crescimento abrupto — um spike de contaminação rápida tende a ter taxa_max alta.

**`taxa_media`**
Velocidade média de variação ao longo de toda a janela. Se positiva, o ferro estava subindo em média; se negativa, caindo. Distingue janelas com tendência geral de acumulação vs. estabilização.

---

### Tendência Global

**`slope`**
Inclinação da reta ajustada por regressão linear sobre toda a janela. Diferente da taxa_media (que é média das derivadas ponto a ponto), o slope captura a **tendência global suavizada**, menos sensível a oscilações locais. Um slope positivo e crescente antes das trocas seria um sinal clássico de degradação progressiva.

---

### Magnitude e Acumulação

**`area_curva`**
Integral do sinal ao longo do tempo (trapézio). Representa a **exposição acumulada** de ferro no período — não só o pico, mas quanto tempo o processo ficou com ferro elevado. Dois eventos podem ter o mesmo ppm_max mas área muito diferente se um foi um pico rápido e outro foi uma contaminação sustentada.

**`ppm_max`** / **`ppm_min`** / **`ppm_media`** / **`ppm_std`**
Estatísticas descritivas clássicas do sinal na janela. O `ppm_std` é especialmente relevante: alta dispersão pode indicar instabilidade do processo, não necessariamente contaminação real.

---

### Comportamento Recente (Último Movimento)

**`ultimo_dy`**
Diferença de ppm entre as duas últimas medições da janela — o movimento imediatamente antes do evento. Captura se o ferro estava subindo ou caindo no momento mais próximo da troca.

**`ultimo_dt`**
Intervalo de tempo entre as duas últimas medições. Junto com `ultimo_dy`, permite reconstruir a taxa final: `ultimo_dy / ultimo_dt`. Útil para saber se o último movimento foi rápido ou lento.

**`ema_final`**
Média móvel exponencial com span igual ao tamanho da janela, avaliada no último ponto. É o **nível suavizado mais recente** do sinal, dando mais peso às medições mais próximas do evento. Filtra ruído e captura se o sinal estava em patamar alto ao se aproximar da troca.

---

### Complexidade do Sinal

**`inversoes_tendencia`**
Número de vezes que a direção do sinal mudou (subida → descida ou vice-versa) entre medições consecutivas. Um sinal oscilatório tem muitas inversões; uma tendência monotônica tem poucas. Captura se o comportamento antes da troca é errático/instável ou progressivo/direcional.

> Contaminação real pode ter poucas inversões (subida contínua), enquanto falsos positivos podem ser picos isolados com muitas inversões.

---

### Análise de Picos

**`crest_factor`**
Razão entre o valor absoluto máximo e o RMS (raiz da média dos quadrados). Mede o quão **extremo** é o pico em relação ao nível médio de energia do sinal. Crest factor alto = pico isolado muito acima do nível base. Crest factor baixo = sinal uniformemente alto.

**`max_prominence`**
A maior proeminência entre todos os picos encontrados na janela. Proeminência mede o quanto um pico se destaca em relação aos vales ao redor — é mais robusto que simplesmente o valor máximo porque desconta o nível base local.

> `crest_factor` e `max_prominence` são complementares: o primeiro é global (relativo ao RMS), o segundo é local (relativo ao entorno do pico).

---

### Extração de Features

#### Baseline features

In [ ]:
def extract_features(df_window):
    """Extrai características da janela para detecção de anomalias."""
    if len(df_window) < 2:
        return None # Ignora janelas com menos de 2 pontos

    # Tempo em horas (relativo ao início da janela) para cálculos matemáticos
    t_hours = (df_window['TIMESTAMP'] - df_window['TIMESTAMP'].min()).dt.total_seconds() / 3600.0
    y_ppm = df_window['Resultado de Ferro (ppm)'].values

    # Derivadas (Taxa de variação ppm/hora)
    dt = np.diff(t_hours)
    dy = np.diff(y_ppm)
    # Evita divisão por zero se houver amostras no exato mesmo segundo
    rates = np.divide(dy, dt, out=np.zeros_like(dy), where=dt!=0) 

    # Inclinação Linear (Slope)
    lr = LinearRegression().fit(t_hours.values.reshape(-1, 1), y_ppm)
    slope = lr.coef_[0]

    # Integral (Área sob a curva)
    area = trapezoid(y=y_ppm, x=t_hours)

    # Foco no Tempo Recente (Último movimento antes do alarme)
    last_dy = dy[-1] if len(dy) > 0 else 0
    last_dt = dt[-1] if len(dt) > 0 else 0
    ema_final = df_window['Resultado de Ferro (ppm)'].ewm(span=len(df_window), adjust=False).mean().iloc[-1]

    # Complexidade e Previsibilidade (Ruído vs Tendência)
    # np.sign retorna direção (-1, 0, 1). np.diff != 0 conta quantas vezes a direção mudou.
    inversoes_tendencia = np.sum(np.diff(np.sign(rates)) != 0) if len(rates) > 1 else 0

    # Análise de Picos e Valores Extremos
    rms = np.sqrt(np.mean(y_ppm**2))
    crest_factor = np.max(np.abs(y_ppm)) / rms if rms > 0 else 0

    # Encontra picos e extrai a maior proeminência (destaque do pico em relação à base)
    picos, propriedades = find_peaks(y_ppm, prominence=0)
    max_prominence = np.max(propriedades['prominences']) if len(picos) > 0 else 0
    return {
        'taxa_max': np.max(rates) if len(rates) > 0 else 0,
        'taxa_media': np.mean(rates) if len(rates) > 0 else 0,
        'slope': slope,
        'area_curva': area,
        'ppm_max': np.max(y_ppm),
        'ppm_min': np.min(y_ppm),
        'ppm_media': np.mean(y_ppm),
        'ppm_std': np.std(y_ppm),
        'ultimo_dy': last_dy,
        'ultimo_dt': last_dt,
        'ema_final': ema_final,
        'inversoes_tendencia': inversoes_tendencia,
        'crest_factor': crest_factor,
        'max_prominence': max_prominence
    }
features_list = []
valid_events = []
eventos_timestamps = df_eventos["TIMESTAMP"]
# Extração de Features por Janela
for evento in eventos_timestamps:
    inicio_janela = evento - pd.Timedelta(days=DIAS_JANELA)

    # Filtra os dados apenas para a janela ANTES do evento
    mask = (df_dataset['TIMESTAMP'] >= inicio_janela) & (df_dataset['TIMESTAMP'] < evento)
    df_window = df_dataset[mask]

    feats = extract_features(df_window)
    if feats:
        features_list.append(feats)
        valid_events.append(evento)
# Criação do DataFrame de Features
df_features = pd.DataFrame(features_list, index=valid_events)
# Adiciona coluna 'Real' do df_eventos
df_real = df_eventos.set_index("TIMESTAMP")["Real"]
df_features["Real"] = df_real.reindex(df_features.index)


# Isolation Forest Score como feature
feature_cols = [col for col in df_features.columns if col != "Real"]
X = df_features[feature_cols].values
y = df_features["Real"].values
# Treina apenas nos exemplos negativos (sem anomalia real)
X_negativos = X[y == 0]
iso = IsolationForest(contamination=0.05, random_state=42)
iso.fit(X_negativos)
# Adiciona o score como feature extra (mais negativo = mais anômalo)
df_features['iso_score'] = iso.decision_function(X)


df_features

#### Features teste

In [ ]:
# def extract_features(df_window, df_baseline=None):
#     """
#     Extrai características da janela para detecção de anomalias.

#     Parâmetros
#     ----------
#     df_window   : DataFrame com colunas TIMESTAMP e 'Resultado de Ferro (ppm)'
#                   contendo os dados da janela anterior ao evento.
#     df_baseline : DataFrame com o mesmo schema, contendo o período de
#                   referência histórica (30 dias antes da janela).
#                   Se None, as features de contexto histórico são omitidas.
#     """
#     if len(df_window) < 2:
#         return None

#     # Vetores base 
#     t_hours = (df_window['TIMESTAMP'] - df_window['TIMESTAMP'].min()) \
#                 .dt.total_seconds() / 3600.0
#     y_ppm   = df_window['Resultado de Ferro (ppm)'].values

#     # Derivadas (taxa de variação ppm/hora) 
#     dt    = np.diff(t_hours)
#     dy    = np.diff(y_ppm)
#     rates = np.divide(dy, dt, out=np.zeros_like(dy), where=dt != 0)

#     # Inclinação linear global 
#     lr    = LinearRegression().fit(t_hours.values.reshape(-1, 1), y_ppm)
#     slope = lr.coef_[0]

#     # Integral 
#     area = trapezoid(y=y_ppm, x=t_hours)

#     # Último movimento antes do evento 
#     last_dy  = dy[-1]  if len(dy) > 0 else 0
#     last_dt  = dt[-1]  if len(dt) > 0 else 0
#     ema_final = df_window['Resultado de Ferro (ppm)'] \
#                     .ewm(span=len(df_window), adjust=False).mean().iloc[-1]

#     # Complexidade / inversões de tendência 
#     inversoes_tendencia = int(np.sum(np.diff(np.sign(rates)) != 0)) \
#                           if len(rates) > 1 else 0

#     # Picos e valores extremos 
#     rms          = np.sqrt(np.mean(y_ppm ** 2))
#     crest_factor = np.max(np.abs(y_ppm)) / rms if rms > 0 else 0

#     picos, props = find_peaks(y_ppm, prominence=0)
#     max_prominence = float(np.max(props['prominences'])) if len(picos) > 0 else 0

#     # Shape da distribuição 
#     ppm_skewness = float(skew(y_ppm))
#     ppm_kurtosis = float(kurtosis(y_ppm))
#     range_norm   = (np.max(y_ppm) - np.min(y_ppm)) / (np.mean(y_ppm) + 1e-9)

#     # Tendência por metades da janela 
#     mid                  = len(y_ppm) // 2
#     media_primeira_metade = float(np.mean(y_ppm[:mid])) if mid >= 1 else float(np.mean(y_ppm))
#     media_segunda_metade  = float(np.mean(y_ppm[mid:])) if mid >= 1 else float(np.mean(y_ppm))
#     ratio_metades         = media_segunda_metade / (media_primeira_metade + 1e-9)

#     # Slope apenas na segunda metade (tendência recente)
#     t_second = t_hours.values[mid:]
#     y_second = y_ppm[mid:]
#     if len(t_second) >= 2:
#         lr_rec       = LinearRegression().fit(t_second.reshape(-1, 1), y_second)
#         slope_recente = float(lr_rec.coef_[0])
#     else:
#         slope_recente = slope  # fallback para slope global

#     # Regularidade temporal da amostragem 
#     n_amostras   = len(y_ppm)
#     dt_medio     = float(np.mean(dt))   if len(dt) > 0 else 0
#     dt_min       = float(np.min(dt))    if len(dt) > 0 else 0
#     dt_variancia = float(np.var(dt))    if len(dt) > 1 else 0
#     duracao_total = float(t_hours.max()) if len(t_hours) > 0 else 1e-9
#     freq_amostras = n_amostras / (duracao_total + 1e-9)

#     # Features de contexto histórico (requerem df_baseline) 
#     if df_baseline is not None and len(df_baseline) >= 2:
#         y_base         = df_baseline['Resultado de Ferro (ppm)'].values
#         baseline_media = float(np.mean(y_base))
#         baseline_std   = float(np.std(y_base))

#         zscore_max   = (np.max(y_ppm)  - baseline_media) / (baseline_std + 1e-9)
#         zscore_media = (np.mean(y_ppm) - baseline_media) / (baseline_std + 1e-9)
#         ratio_media  = np.mean(y_ppm)  / (baseline_media + 1e-9)
#         ratio_max    = np.max(y_ppm)   / (baseline_media + 1e-9)

#         # Threshold = percentil 75 do baseline
#         threshold_hist     = float(np.percentile(y_base, 75))
#         n_acima_threshold  = int(np.sum(y_ppm > threshold_hist))
#         pct_acima_threshold = float(np.mean(y_ppm > threshold_hist))

#         # Maior sequência consecutiva acima do threshold
#         flags = np.concatenate([[False], y_ppm > threshold_hist, [False]])
#         diffs = np.diff(flags.astype(int))
#         starts = np.where(diffs == 1)[0]
#         ends   = np.where(diffs == -1)[0]
#         max_run_acima = int(np.max(ends - starts)) if len(starts) > 0 else 0

#     else:
#         # Sem baseline disponível — preenche com NaN para não distorcer o modelo
#         zscore_max = zscore_media = ratio_media = ratio_max = np.nan
#         n_acima_threshold = pct_acima_threshold = max_run_acima = np.nan

#     # Dicionário final 
#     return {
#         # Taxas de variação
#         'taxa_max':               float(np.max(rates)) if len(rates) > 0 else 0,
#         'taxa_media':             float(np.mean(rates)) if len(rates) > 0 else 0,
#         # Tendência global
#         'slope':                  float(slope),
#         'slope_recente':          slope_recente,
#         # Magnitude e acumulação
#         'area_curva':             float(area),
#         'ppm_max':                float(np.max(y_ppm)),
#         'ppm_min':                float(np.min(y_ppm)),
#         'ppm_media':              float(np.mean(y_ppm)),
#         'ppm_std':                float(np.std(y_ppm)),
#         # Último movimento
#         'ultimo_dy':              float(last_dy),
#         'ultimo_dt':              float(last_dt),
#         'ema_final':              float(ema_final),
#         # Complexidade
#         'inversoes_tendencia':    inversoes_tendencia,
#         # Picos
#         'crest_factor':           float(crest_factor),
#         'max_prominence':         max_prominence,
#         # Shape da distribuição
#         'skewness':               ppm_skewness,
#         'kurtosis':               ppm_kurtosis,
#         'range_norm':             float(range_norm),
#         # Tendência por metades
#         'media_primeira_metade':  media_primeira_metade,
#         'media_segunda_metade':   media_segunda_metade,
#         'ratio_metades':          float(ratio_metades),
#         # Amostragem
#         'n_amostras':             n_amostras,
#         'dt_medio':               dt_medio,
#         'dt_min':                 dt_min,
#         'dt_variancia':           dt_variancia,
#         'freq_amostras':          float(freq_amostras),
#         # Contexto histórico
#         'zscore_max':             zscore_max,
#         'zscore_media':           zscore_media,
#         'ratio_media':            ratio_media,
#         'ratio_max':              ratio_max,
#         'n_acima_threshold':      n_acima_threshold,
#         'pct_acima_threshold':    pct_acima_threshold,
#         'max_run_acima':          max_run_acima,
#     }


# # EXTRAÇÃO DE FEATURES POR JANELA
# features_list    = []
# valid_events     = []
# eventos_timestamps = df_eventos["TIMESTAMP"]

# for evento in eventos_timestamps:
#     inicio_janela   = evento - pd.Timedelta(days=DIAS_JANELA)
#     inicio_baseline = evento - pd.Timedelta(days=DIAS_JANELA + DIAS_BASELINE)

#     # Janela principal (7 dias antes do evento)
#     mask_window = (
#         (df_dataset['TIMESTAMP'] >= inicio_janela) &
#         (df_dataset['TIMESTAMP'] <  evento)
#     )
#     df_window = df_dataset[mask_window]

#     # Janela de baseline histórico (30 dias antes da janela principal)
#     mask_baseline = (
#         (df_dataset['TIMESTAMP'] >= inicio_baseline) &
#         (df_dataset['TIMESTAMP'] <  inicio_janela)
#     )
#     df_base = df_dataset[mask_baseline]

#     # Passa df_base=None se não houver dados suficientes no baseline
#     baseline_arg = df_base if len(df_base) >= 2 else None

#     feats = extract_features(df_window, df_baseline=baseline_arg)
#     if feats:
#         features_list.append(feats)
#         valid_events.append(evento)

# df_features = pd.DataFrame(features_list, index=valid_events)
# # Adiciona coluna 'Real' do df_eventos
# df_real = df_eventos.set_index("TIMESTAMP")["Real"]
# df_features["Real"] = df_real.reindex(df_features.index)

# # Preenche NaN das features de baseline com a mediana da coluna (ocorre quando não há dados suficientes no período de referência)
# cols_baseline = [
#     'zscore_max', 'zscore_media', 'ratio_media', 'ratio_max',
#     'n_acima_threshold', 'pct_acima_threshold', 'max_run_acima'
# ]
# for col in cols_baseline:
#     if df_features[col].isna().any():
#         df_features[col] = df_features[col].fillna(df_features[col].median())

# # ISOLATION FOREST SCORE COMO FEATURE
# feature_cols = [col for col in df_features.columns if col != "Real"]
# X = df_features[feature_cols].values
# y = df_features["Real"].values

# X_negativos = X[y == 0]
# iso = IsolationForest(contamination=0.05, random_state=42)
# iso.fit(X_negativos)

# df_features['iso_score'] = iso.decision_function(X)

# df_features

### Escalonamento dos dados

In [ ]:
# Padronização e Clusterização
scaler = StandardScaler() #StandardScaler || RobustScaler || MinMaxScaler
X_scaled = scaler.fit_transform(df_features.drop('Real', axis=1))
y = df_features['Real']

### Verificando separabilidade das amostras com PCA

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

cores = {0: 'steelblue', 1: 'crimson'}
labels_texto = {0: 'Sem Evento apontado', 1: 'Evento apontado'}

fig, ax = plt.subplots(figsize=(8, 6))
for val in [0, 1]:
    mask = df_features['Real'] == val
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=cores[val], label=labels_texto[val],
               s=100, edgecolors='k', linewidths=0.8)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variância)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variância)')
ax.legend()
ax.set_title('Separabilidade das classes')
plt.tight_layout()
plt.show()

## KMeans

In [ ]:
# Elbow Method
inercia = []
K_range = range(1, 25)

for k in K_range:
    kmeans_teste = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_teste.fit(X_scaled)
    inercia.append(kmeans_teste.inertia_)

# Plota o gráfico para visualização
plt.figure(figsize=(10,5))
plt.plot(K_range, inercia, marker='o', linestyle='--')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia')
plt.title('Método do Cotovelo para K Ideal')
plt.xticks(K_range)
plt.grid(True)
plt.show()

In [ ]:
# Clusterização
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10) # Como são duas classes, o número de clusters foi fixado como 2
df_features['Cluster'] = kmeans.fit_predict(X_scaled)
# Resultado final
df_features

In [ ]:
# Dicionário de cores dos clusters
cores_clusters = {
    0: 'blue',   
    1: 'green'   
}

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_dataset['TIMESTAMP'],
    y=df_dataset["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos.iterrows():
    evento_ts = row["TIMESTAMP"]
    
    # Adiciona o sombreado da janela baseado no Cluster
    if evento_ts in df_features.index:
        cluster = df_features.loc[evento_ts, 'Cluster']
        cor_fundo = cores_clusters.get(cluster, 'gray')
        inicio_janela = evento_ts - pd.Timedelta(days=DIAS_JANELA)
        
        fig.add_vrect(
            x0=inicio_janela,
            x1=evento_ts,
            fillcolor=cor_fundo,
            opacity=0.2, 
            layer="below", 
            line_width=0,
            annotation_text=f"C{cluster}",
            annotation_position="top left"
        )

    # Linha vertical exata do evento
    cor = "red" if row["Real"] == 1 else "blue"
    fig.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )

    # Anotação
    fig.add_annotation(
        x=str(evento_ts),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title='Teor de Fe - Clusters'
)
fig.show()

In [ ]:
# Tabela de contingência
pd.crosstab(df_features['Cluster'], df_features['Real'], 
            rownames=['Cluster'], colnames=['Real'])

In [ ]:
## Quão bem os clusters recuperam os labels reais
ari  = adjusted_rand_score(df_features['Real'], df_features['Cluster'])
nmi  = normalized_mutual_info_score(df_features['Real'], df_features['Cluster'])

print(f"Adjusted Rand Score: {ari}")
print(f"Normalized Mutual Info Score: {ari}")

## Hierarchical Clustering

In [ ]:
Z = linkage(X_scaled, method='ward')

fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle(f'Dendrograma', fontsize=13, fontweight='bold')

dendrogram(Z, ax=ax, color_threshold=0, leaf_rotation=90, leaf_font_size=7)

ax.set_xlabel('Índice da Amostra')
ax.set_ylabel('Distância de Fusão')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Clusterização
hc_final = AgglomerativeClustering(n_clusters=2, linkage='ward')
df_features['Cluster'] = hc_final.fit_predict(X_scaled)

df_features

In [ ]:
pd.crosstab(
    df_features['Cluster'], df_features['Real'],
    rownames=['Cluster'], colnames=['Real'],
)

In [ ]:
## Quão bem os clusters recuperam os labels reais
ari  = adjusted_rand_score(df_features['Real'], df_features['Cluster'])
nmi  = normalized_mutual_info_score(df_features['Real'], df_features['Cluster'])

print(f"Adjusted Rand Score: {ari}")
print(f"Normalized Mutual Info Score: {ari}")

In [ ]:
# Dicionário de cores dos clusters
cores_clusters = {
    0: 'blue',   
    1: 'green'   
}

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_dataset['TIMESTAMP'],
    y=df_dataset["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos.iterrows():
    evento_ts = row["TIMESTAMP"]
    
    # Adiciona o sombreado da janela baseado no Cluster
    if evento_ts in df_features.index:
        cluster = df_features.loc[evento_ts, 'Cluster']
        cor_fundo = cores_clusters.get(cluster, 'gray')
        inicio_janela = evento_ts - pd.Timedelta(days=DIAS_JANELA)
        
        fig.add_vrect(
            x0=inicio_janela,
            x1=evento_ts,
            fillcolor=cor_fundo,
            opacity=0.2, 
            layer="below", 
            line_width=0,
            annotation_text=f"C{cluster}",
            annotation_position="top left"
        )

    # Linha vertical exata do evento
    cor = "red" if row["Real"] == 1 else "blue"
    fig.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )

    # Anotação
    fig.add_annotation(
        x=str(evento_ts),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title='Teor de Fe - Clusters'
)
fig.show()

## ROCKET

## kShape

# Classificação (Abordagem Supervisionada)

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
import numpy as np

X = df_features.drop(columns=['real'])
y = df_features['real']

# scale_pos_weight = n_negativos / n_positivos
modelo = XGBClassifier(
    scale_pos_weight=83/9,  # corrige o desbalanceamento
    n_estimators=100,
    max_depth=3,            # raso — evita overfitting com 92 amostras
    learning_rate=0.05,
    subsample=0.8,
    eval_metric='aucpr',    # AUC da curva Precision-Recall
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Stratified garante que cada fold tenha proporção de positivos

scores = cross_validate(modelo, X, y, cv=cv, scoring={
    'f1':      make_scorer(f1_score),
    'roc_auc': 'roc_auc',
    'pr_auc':  'average_precision'
})

print(f"F1:     {scores['test_f1'].mean():.3f} ± {scores['test_f1'].std():.3f}")
print(f"ROCAUC: {scores['test_roc_auc'].mean():.3f} ± {scores['test_roc_auc'].std():.3f}")
print(f"PR-AUC: {scores['test_pr_auc'].mean():.3f} ± {scores['test_pr_auc'].std():.3f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler

pipeline = ImbPipeline([
    ('scaler',  StandardScaler()),
    ('smote',   SMOTE(k_neighbors=3, random_state=42)),
    # k_neighbors=3 porque só temos 9 positivos
    ('modelo',  LogisticRegression(class_weight='balanced', max_iter=1000))
])

scores = cross_validate(pipeline, X, y, cv=cv, scoring={
    'f1':     make_scorer(f1_score),
    'pr_auc': 'average_precision'
})

In [ ]:
from sklearn.metrics import precision_recall_curve

modelo.fit(X, y)
probs = modelo.predict_proba(X)[:, 1]

precision, recall, thresholds = precision_recall_curve(y, probs)

# Threshold que maximiza F1
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Threshold ótimo: {best_threshold:.3f}")
```

---

## Métricas para Usar e Evitar

| Métrica | Usar? | Motivo |
|---|---|---|
| Acurácia | ❌ | Enganosa com desbalanceamento |
| ROC-AUC | ⚠️ Cuidado | Otimista com classes desbalanceadas |
| **PR-AUC** | ✅ Principal | Honesta com desbalanceamento |
| **F1-score** | ✅ Principal | Balanceia precisão e recall |
| **Recall dos positivos** | ✅ Crítico | Falso negativo (perder contaminação real) é o pior erro |
| Precision dos positivos | ✅ Secundário | Falso alarme tem custo menor que parar desnecessariamente |

---

## Arquitetura Final Sugerida
```
92 janelas de 7 dias (features extraídas)
          │
          ├── Clustering (sem labels)
          │     └─ Validar com ARI/NMI → as features fazem sentido?
          │
          ├── Isolation Forest nos 83 negativos
          │     └─ iso_score como feature extra
          │
          ├── XGBoost com scale_pos_weight=9.2
          │     └─ StratifiedKFold(5) + PR-AUC + F1
          │     └─ Threshold tuning pela curva PR
          │
          └── SHAP values para interpretabilidade
                └─ Quais features mais separam Real=1 de Real=0?
                └─ Apresentar para equipe de manutenção

## LDA/QDA

In [ ]:
"""
Análise Discriminante Linear (LDA) e Quadrática (QDA)
para detecção de contaminação de ferro em ppm.

Contexto: 9 eventos reais (Real=1) vs 83 falsos positivos (Real=0)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_validate, LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay, make_scorer, f1_score
)
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

# =============================================================================
# CONFIGURAÇÕES
# =============================================================================

RANDOM_STATE  = 42
N_SPLITS_CV   = 5          # folds para StratifiedKFold
REG_PARAM_QDA = 1e-2       # regularização QDA (0 = sem reg, aumentar se singular)

X = df_features.drop('Real', axis=1)
y = df_features['Real']
feature_names = df_features.drop(columns=['Real']).columns.tolist()

print("=" * 60)
print("DISTRIBUIÇÃO DAS CLASSES")
print("=" * 60)
print(f"  Real=0 (falso positivo): {(y==0).sum()} amostras")
print(f"  Real=1 (contaminação):   {(y==1).sum()} amostras")
print(f"  Razão desbalanceamento:  1:{(y==0).sum()//(y==1).sum()}")
print()

# =============================================================================
# 2. DEFINIÇÃO DOS MODELOS
# =============================================================================

modelos = {
    'LDA': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'))
        # shrinkage='auto' usa Ledoit-Wolf — robusto com poucas amostras
    ]),
    'QDA': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    QuadraticDiscriminantAnalysis(reg_param=REG_PARAM_QDA))
        # reg_param regulariza a matriz de covariância — necessário com n pequeno
    ])
}

# =============================================================================
# 3. VALIDAÇÃO CRUZADA
# =============================================================================

print("=" * 60)
print("VALIDAÇÃO CRUZADA — StratifiedKFold (5 folds)")
print("=" * 60)

cv_strat = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)
cv_loo   = LeaveOneOut()

scoring = {
    'f1':       make_scorer(f1_score, zero_division=0),
    'roc_auc':  'roc_auc',
    'pr_auc':   'average_precision',
    'recall':   make_scorer(__import__('sklearn.metrics', fromlist=['recall_score'])
                            .recall_score, zero_division=0),
    'precision': make_scorer(__import__('sklearn.metrics', fromlist=['precision_score'])
                             .precision_score, zero_division=0),
}

resultados_cv = {}

for nome, pipeline in modelos.items():
    scores = cross_validate(pipeline, X, y, cv=cv_strat, scoring=scoring)
    resultados_cv[nome] = scores

    print(f"\n  {nome}")
    print(f"    F1-score  : {scores['test_f1'].mean():.3f} ± {scores['test_f1'].std():.3f}")
    print(f"    ROC-AUC   : {scores['test_roc_auc'].mean():.3f} ± {scores['test_roc_auc'].std():.3f}")
    print(f"    PR-AUC    : {scores['test_pr_auc'].mean():.3f} ± {scores['test_pr_auc'].std():.3f}")
    print(f"    Recall    : {scores['test_recall'].mean():.3f} ± {scores['test_recall'].std():.3f}")
    print(f"    Precision : {scores['test_precision'].mean():.3f} ± {scores['test_precision'].std():.3f}")

# =============================================================================
# 4. LEAVE-ONE-OUT — validação complementar (n pequeno)
# =============================================================================

print("\n" + "=" * 60)
print("LEAVE-ONE-OUT (LOO) — validação complementar")
print("=" * 60)

for nome, pipeline in modelos.items():
    scores_loo = cross_validate(pipeline, X, y, cv=cv_loo,
                                scoring={'f1': make_scorer(f1_score, zero_division=0),
                                         'roc_auc': 'roc_auc'})
    print(f"\n  {nome} (LOO)")
    print(f"    F1-score : {scores_loo['test_f1'].mean():.3f}")
    print(f"    ROC-AUC  : {scores_loo['test_roc_auc'].mean():.3f}")

# =============================================================================
# 5. AJUSTE FINAL E THRESHOLD TUNING
# =============================================================================

print("\n" + "=" * 60)
print("THRESHOLD TUNING — otimização pelo F1 na curva PR")
print("=" * 60)

modelos_fit   = {}
thresholds_ot = {}

for nome, pipeline in modelos.items():
    pipeline.fit(X, y)
    modelos_fit[nome] = pipeline

    probs = pipeline.predict_proba(X)[:, 1]
    precision_arr, recall_arr, thresh_arr = precision_recall_curve(y, probs)

    f1_arr  = np.where(
        (precision_arr + recall_arr) > 0,
        2 * precision_arr * recall_arr / (precision_arr + recall_arr),
        0
    )
    best_idx     = np.argmax(f1_arr)
    best_thresh  = thresh_arr[best_idx] if best_idx < len(thresh_arr) else 0.5
    thresholds_ot[nome] = best_thresh

    print(f"\n  {nome}")
    print(f"    Threshold ótimo : {best_thresh:.3f}")
    print(f"    F1 no threshold : {f1_arr[best_idx]:.3f}")

# =============================================================================
# 6. RELATÓRIO DE CLASSIFICAÇÃO COM THRESHOLD OTIMIZADO
# =============================================================================

print("\n" + "=" * 60)
print("RELATÓRIO DE CLASSIFICAÇÃO (threshold otimizado, treino completo)")
print("=" * 60)

for nome, pipeline in modelos_fit.items():
    probs  = pipeline.predict_proba(X)[:, 1]
    y_pred = (probs >= thresholds_ot[nome]).astype(int)

    print(f"\n  {nome} — threshold={thresholds_ot[nome]:.3f}")
    print(classification_report(y, y_pred,
                                 target_names=['Falso Positivo', 'Contaminação Real'],
                                 zero_division=0))

# =============================================================================
# 7. IMPORTÂNCIA DE FEATURES (permutation importance)
# =============================================================================

print("=" * 60)
print("IMPORTÂNCIA DE FEATURES — permutation importance")
print("=" * 60)

importancias = {}
for nome, pipeline in modelos_fit.items():
    perm = permutation_importance(
        pipeline, X, y,
        n_repeats=30,
        random_state=RANDOM_STATE,
        scoring='average_precision'
    )
    importancias[nome] = perm
    idx_sorted = np.argsort(perm.importances_mean)[::-1]
    print(f"\n  {nome} — top 5 features:")
    for i in idx_sorted[:5]:
        print(f"    {feature_names[i]:<25} {perm.importances_mean[i]:.4f} ± {perm.importances_std[i]:.4f}")

# =============================================================================
# 8. VISUALIZAÇÕES
# =============================================================================

fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

cores_classe = {0: '#4878CF', 1: '#D65F5F'}
nomes_classe = {0: 'Falso Positivo', 1: 'Contaminação Real'}

# ── 8.1  Matrizes de Confusão ──────────────────────────────────────────────
for col, (nome, pipeline) in enumerate(modelos_fit.items()):
    ax = fig.add_subplot(gs[0, col])
    probs  = pipeline.predict_proba(X)[:, 1]
    y_pred = (probs >= thresholds_ot[nome]).astype(int)
    cm = confusion_matrix(y, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Falso +', 'Real'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{nome}\nMatriz de Confusão (thr={thresholds_ot[nome]:.2f})',
                 fontsize=11, fontweight='bold')

# ── 8.2  Curvas ROC ────────────────────────────────────────────────────────
ax_roc = fig.add_subplot(gs[0, 2])
for nome, pipeline in modelos_fit.items():
    probs = pipeline.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y, probs)
    roc_auc_val = auc(fpr, tpr)
    ax_roc.plot(fpr, tpr, lw=2, label=f'{nome} (AUC={roc_auc_val:.3f})')
ax_roc.plot([0,1],[0,1],'k--', lw=1, label='Aleatório')
ax_roc.set_xlabel('Taxa Falso Positivo'); ax_roc.set_ylabel('Taxa Verdadeiro Positivo')
ax_roc.set_title('Curva ROC', fontweight='bold')
ax_roc.legend(fontsize=9); ax_roc.grid(alpha=0.3)

# ── 8.3  Curvas Precision-Recall ───────────────────────────────────────────
ax_pr = fig.add_subplot(gs[1, 0])
for nome, pipeline in modelos_fit.items():
    probs = pipeline.predict_proba(X)[:, 1]
    prec, rec, _ = precision_recall_curve(y, probs)
    ap = average_precision_score(y, probs)
    ax_pr.plot(rec, prec, lw=2, label=f'{nome} (AP={ap:.3f})')
ax_pr.axhline(y=(y==1).mean(), color='k', linestyle='--', lw=1, label='Baseline')
ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.set_title('Curva Precision-Recall\n(mais informativa com desbalanceamento)',
                fontweight='bold')
ax_pr.legend(fontsize=9); ax_pr.grid(alpha=0.3)

# ── 8.4  Scores de probabilidade por classe ────────────────────────────────
for col, (nome, pipeline) in enumerate(modelos_fit.items()):
    ax = fig.add_subplot(gs[1, col + 1])
    probs = pipeline.predict_proba(X)[:, 1]
    for cls in [0, 1]:
        mask = y == cls
        ax.hist(probs[mask], bins=15, alpha=0.6,
                color=cores_classe[cls], label=nomes_classe[cls], edgecolor='white')
    ax.axvline(thresholds_ot[nome], color='black', linestyle='--', lw=1.5,
               label=f'Threshold={thresholds_ot[nome]:.2f}')
    ax.set_xlabel('Probabilidade predita (classe Real=1)')
    ax.set_ylabel('Contagem')
    ax.set_title(f'{nome}\nDistribuição de Scores', fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ── 8.5  Importância de Features ──────────────────────────────────────────
for col, (nome, perm) in enumerate(importancias.items()):
    ax = fig.add_subplot(gs[2, col])
    idx_sorted = np.argsort(perm.importances_mean)[::-1][:10]
    y_pos = np.arange(len(idx_sorted))
    ax.barh(y_pos, perm.importances_mean[idx_sorted], xerr=perm.importances_std[idx_sorted],
            color='steelblue', alpha=0.8, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels([feature_names[i] for i in idx_sorted], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Redução em PR-AUC (permutação)')
    ax.set_title(f'{nome}\nTop 10 Features', fontweight='bold')
    ax.grid(alpha=0.3, axis='x')

# ── 8.6  Comparação CV (F1 por fold) ──────────────────────────────────────
ax_cv = fig.add_subplot(gs[2, 2])
for nome, scores in resultados_cv.items():
    ax_cv.plot(range(1, N_SPLITS_CV + 1), scores['test_f1'],
               marker='o', lw=2, label=nome)
ax_cv.set_xlabel('Fold'); ax_cv.set_ylabel('F1-score')
ax_cv.set_title('F1-score por Fold (StratifiedKFold)', fontweight='bold')
ax_cv.set_xticks(range(1, N_SPLITS_CV + 1))
ax_cv.legend(); ax_cv.grid(alpha=0.3)

fig.suptitle('Análise Discriminante: LDA vs QDA\nDetecção de Contaminação de Ferro',
             fontsize=14, fontweight='bold', y=1.01)

plt.show()

# =============================================================================
# 9. RESUMO FINAL
# =============================================================================

print("\n" + "=" * 60)
print("RESUMO COMPARATIVO")
print("=" * 60)
print(f"{'Modelo':<8} {'F1':>8} {'ROC-AUC':>10} {'PR-AUC':>10} {'Recall':>9} {'Threshold':>11}")
print("-" * 60)
for nome, scores in resultados_cv.items():
    print(f"{nome:<8} "
          f"{scores['test_f1'].mean():>8.3f} "
          f"{scores['test_roc_auc'].mean():>10.3f} "
          f"{scores['test_pr_auc'].mean():>10.3f} "
          f"{scores['test_recall'].mean():>9.3f} "
          f"{thresholds_ot[nome]:>11.3f}")